Hospital Managament System

db.py

In [ ]:
import sqlite3

def get_db():
    con = sqlite3.connect("hospital.db")
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS patients (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT,
            age INTEGER,
            department TEXT,
            doctor TEXT,
            token INTEGER,
            time TEXT,
            status TEXT
        )
    """)
    con.commit()
    return con


models.py

In [ ]:
import db

def add_patient(name, age, department, doctor, token, time):
    con = db.get_db()
    cur = con.cursor()
    cur.execute(
        "INSERT INTO patients (name, age, department, doctor, token, time, status) VALUES (?,?,?,?,?,?,?)",
        (name, age, department, doctor, token, time, "Waiting")
    )
    con.commit()
    con.close()

def get_last_token(doctor):
    con = db.get_db()
    cur = con.cursor()
    cur.execute(
        "SELECT MAX(token) FROM patients WHERE doctor=?",
        (doctor,)
    )
    t = cur.fetchone()[0]
    con.close()
    return t if t else 0

def get_queue(doctor):
    con = db.get_db()
    cur = con.cursor()
    cur.execute(
        "SELECT token, name, time, status FROM patients WHERE doctor=? ORDER BY token",
        (doctor,)
    )
    rows = cur.fetchall()
    con.close()
    return rows

def mark_consulted(token):
    con = db.get_db()
    cur = con.cursor()
    cur.execute(
        "UPDATE patients SET status='Consulted' WHERE token=?",
        (token,)
    )
    con.commit()
    con.close()

def search_patient(value):
    con = db.get_db()
    cur = con.cursor()
    cur.execute(
        "SELECT * FROM patients WHERE name LIKE ? OR token=?",
        (f"%{value}%", value if value.isdigit() else -1)
    )
    rows = cur.fetchall()
    con.close()
    return rows
def get_ongoing_consultations():
    con = db.get_db()
    cur = con.cursor()
    cur.execute(
        "SELECT doctor, token, name, time FROM patients WHERE status='Waiting' ORDER BY doctor, token"
    )
    rows = cur.fetchall()
    con.close()
    return rows


utils.py

In [ ]:
from datetime import datetime, timedelta

def estimate_time(token):
    start = datetime.strptime("10:00", "%H:%M")
    return (start + timedelta(minutes=10 * (token - 1))).strftime("%H:%M")


requirements.txt

In [ ]:
# no external packages required; uses Python stdlib and sqlite3

main.py

In [ ]:
from models import (
    add_patient,
    get_last_token,
    get_queue,
    mark_consulted,
    search_patient,
    get_ongoing_consultations
)
from utils import estimate_time

doctors = {
    "1": "Dr. Sharma",
    "2": "Dr. Mehta",
    "3": "Dr. Singh"
}

while True:
    print("\nHOSPITAL APPOINTMENT SYSTEM")
    print("1 Register Patient")
    print("2 View Doctor Queue")
    print("3 Mark Consultation Done")
    print("4 Search Patient")
    print("5 View Ongoing Consultations")
    print("6 Exit")

    ch = input("Choose option: ")

    if ch == "1":
        name = input("Patient name: ")
        age = int(input("Age: "))
        dept = input("Department: ")

        print("Choose Doctor:")
        for k, v in doctors.items():
            print(k, v)

        d = input("Enter doctor number: ")

        if d not in doctors:
            print("Invalid doctor selection")
            continue

        doctor = doctors[d]
        token = get_last_token(doctor) + 1
        time = estimate_time(token)

        add_patient(name, age, dept, doctor, token, time)
        print(f"Registered | Token {token} | Time {time}")

    elif ch == "2":
        print("Choose Doctor:")
        for k, v in doctors.items():
            print(k, v)

        d = input("Enter doctor number: ")

        if d not in doctors:
            print("Invalid doctor selection")
            continue

        doctor = doctors[d]
        queue = get_queue(doctor)

        print(f"\nQueue for {doctor}")
        for q in queue:
            print(q)

    elif ch == "3":
        token = int(input("Enter token number: "))
        mark_consulted(token)
        print("Consultation marked complete")

    elif ch == "4":
        val = input("Enter patient name or token: ")
        results = search_patient(val)
        for r in results:
            print(r)

    elif ch == "5":
        ongoing = get_ongoing_consultations()
        if not ongoing:
            print("No ongoing consultations")
        else:
            print("\nOngoing Consultations:")
            for o in ongoing:
                print(o)

    elif ch == "6":
        print("Exiting system")
        break

    else:
        print("Invalid option")


README.md

In [ ]:
# Hospital Appointment & Queue Management System

## Features
- Patient registration
- Doctor-wise token generation
- Appointment time estimation
- Queue management
- Consultation tracking
- Search by name or token

## Tech Stack
Python, SQLite

## How to Run
python main.py
